# 08 · 3D map for ArcGIS Pro / QGIS (as V3 notebook 10)

For the map only, the final model type, hyperparameters and threshold are **refitted on all 120 Downtown boreholes** (109 development + 11 holdout), with V3's seed. This map model is **not evaluated**: its skill is the one estimated by the CV and the external tests.

| Column | Meaning |
|---|---|
| `X, Y, Z` | UTM 48N (EPSG:32648), cell-centre elevation |
| `p_clay`, `clay` | probability of Clay, and the class at the threshold (1 = Clay) |
| `dist_nearest_bh_m`, `reliability_zone` | distance to the nearest of the 120 boreholes; zone 1 = ≤ 100 m, 2 = 100-300 m, 3 = > 300 m |
| `extrap_vertical` | 1 = elevation outside the training data |

**ArcGIS Pro:** *XY Table To Point* with X = `X`, Y = `Y`, Z = `Z`, EPSG:32648; or add the `.nc` file as a voxel layer.

**Run time:** 10-30 minutes.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
rec = json.loads((H.out_dir(S, "05_final_model") / "final_model.json").read_text())
SPACING = S["quick"]["map_spacing_m"] if S["run"]["quick"] else S["map"]["spacing_m"]
OUT, PRIV = H.out_dir(S, "08_map"), H.out_dir(S, "08_map", private=True)
train = cells[cells.set.isin(["development", "external"])]
print(f"Map model: {rec['model']} on {train.BH_Code.nunique()} boreholes, {len(train)} cells | grid {SPACING:g} m")
model = H.refit(rec["model"], rec["params"], train, f"map120_{rec['model']}", S)
path = PRIV / f"map_model_{rec['model']}_120bh.joblib"
joblib.dump(model, path, compress=3)

In [ ]:
cols = H.map_columns(S, SPACING)
grid = H.map_cells(cols, S)
n = S["map"]["chunk_rows"]
p = np.empty(len(grid), dtype="float32")
X = grid[H.FEATURES].values
for i in range(0, len(grid), n):
    p[i:i + n] = model.predict_proba(X[i:i + n])[:, 1]
grid["p_clay"] = p
grid["clay"] = (p >= rec["threshold"]).astype("int8")
xy = train.drop_duplicates("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].values
d = H.nearest_distance(cols[["X", "Y"]].values, xy)
grid["dist_nearest_bh_m"] = d[grid.col].astype("float32")
grid["reliability_zone"] = (np.searchsorted(np.asarray(S["map"]["zone_edges_m"]), grid.dist_nearest_bh_m.values, side="right") + 1).astype("int8")
lo, hi = float(train.Top_Elevation_m.min()), float(train.Top_Elevation_m.max())
grid["extrap_vertical"] = ((grid.Top_Elevation_m < lo) | (grid.Top_Elevation_m > hi)).astype("int8")

In [ ]:
tag = f"{int(SPACING)}m"
grid.rename(columns={"UTM_Easting_m": "X", "UTM_Northing_m": "Y"})[
    ["X", "Y", "Z", "depth_top_m", "Top_Elevation_m", "ground_m", "p_clay", "clay", "dist_nearest_bh_m",
     "reliability_zone", "extrap_vertical"]].to_csv(PRIV / f"siemreap_3d_points_120bh_{tag}.csv", index=False, float_format="%.3f")
vox = H.to_voxel_netcdf(grid, cols, S, PRIV / f"siemreap_3d_voxels_120bh_{tag}.nc")
summary = {"columns": len(cols), "cells": len(grid), "spacing_m": SPACING,
           "reliability_zone_share": grid.reliability_zone.value_counts(normalize=True).sort_index().round(4).to_dict(),
           "vertical_extrapolation": float(grid.extrap_vertical.mean()), "predicted_clay_share": float(grid.clay.mean()),
           "voxels": vox, "model_sha256": H.sha256(path)}
(OUT / f"map_summary_120bh_{tag}.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str))
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 09.")